***Filtrado por equipos, Selección de variables y Eliminación de nulos:***

In [ ]:
import pandas as pd

# Cargamos todos los CSV filtrados
df_2021 = pd.read_csv('../data/filtered/LECdata-2021.csv', low_memory=False)
df_2022 = pd.read_csv('../data/filtered/LECdata-2022.csv', low_memory=False)
df_2023 = pd.read_csv('../data/filtered/LECdata-2023.csv', low_memory=False)
df_2024 = pd.read_csv('../data/filtered/LECdata-2024.csv', low_memory=False)
df_2025 = pd.read_csv('../data/filtered/LECdata-2025.csv', low_memory=False)

# Unificamos todos los datos en una variable
df_historico = pd.concat([df_2021, df_2022, df_2023, df_2024, df_2025], ignore_index=True)


# ----------------- CÁLCULO DE WIN RATIO POR JUGADOR -----------------
# Tomamos las filas correspondientes a jugadores
df_jugadores = df_historico[df_historico['position'] != 'team'].copy()

# Realizamos una media de las victorias anteriores para cada jugador aplicando time decay (EWM)
df_jugadores['player_win_ratio'] = df_jugadores.groupby('playername')['result'].transform(lambda x: x.shift().ewm(span=20, min_periods=1).mean().round(4))
# Contamos la cantidad de partidas previas del jugador
df_jugadores['player_games'] = df_jugadores.groupby('playername')['result'].transform(lambda x: x.shift().expanding().count())

# Para los jugadores con menos de 20 partidas, el win ratio es 0.5 (50%)
df_jugadores['player_win_ratio'] = df_jugadores['player_win_ratio'].where(df_jugadores['player_games'] >= 20, 0.5).fillna(0.5)

# Asociamos el win ratio a cada posicón
df_wr = df_jugadores.pivot_table(index=['gameid', 'teamname'], columns='position', values='player_win_ratio').reset_index()
df_wr = df_wr.rename(columns={
    'top': 'wr_top',
    'jng': 'wr_jng',
    'mid': 'wr_mid',
    'bot': 'wr_bot',
    'sup': 'wr_sup'
})
# ---------------------------------------------------------------------------

# ----------------- MAPEO DE CAMPEONES POR POSICIÓN -----------------
# Asociamos el nombre del campeón a cada posición jugada
df_champs = df_jugadores.pivot(index=['gameid', 'teamname'], columns='position', values='champion').reset_index()
df_champs = df_champs.rename(columns={
    'top': 'champ_top',
    'jng': 'champ_jng',
    'mid': 'champ_mid',
    'bot': 'champ_bot',
    'sup': 'champ_sup'
})
# ---------------------------------------------------------------------------

# ----------------- CÁLCULO DE WIN RATIO POR CAMPEÓN -----------------
# Realizamos una media de las victorias anteriores para cada campeón aplicando time decay (EWM)
df_jugadores['champ_win_ratio'] = df_jugadores.groupby('champion')['result'].transform(lambda x: x.shift().ewm(span=20, min_periods=1).mean().round(4))
# Contamos la cantidad de partidas previas del campeón
df_jugadores['champ_wr_games'] = df_jugadores.groupby('champion')['result'].transform(lambda x: x.shift().expanding().count())

# Para los campeones con menos de 20 partidas, el win ratio es 0.5 (50%)
df_jugadores['champ_win_ratio'] = df_jugadores['champ_win_ratio'].where(df_jugadores['champ_wr_games'] >= 20, 0.5).fillna(0.5)

# Asociamos el win ratio del campeón a cada posición
df_champ_wr = df_jugadores.pivot_table(index=['gameid', 'teamname'], columns='position', values='champ_win_ratio').reset_index()
df_champ_wr = df_champ_wr.rename(columns={
    'top': 'wr_champ_top',
    'jng': 'wr_champ_jng',
    'mid': 'wr_champ_mid',
    'bot': 'wr_champ_bot',
    'sup': 'wr_champ_sup'
})
# ---------------------------------------------------------------------------

# ----------------- CÁLCULO DE COMP EARLY POWER POR EQUIPO -----------------
# Calculamos la media histórica de oro al min 15 para cada campeón aplicando time decay (EWM)
df_jugadores['champ_gold_mean'] = df_jugadores.groupby('champion')['golddiffat15'].transform(lambda x: x.shift().ewm(span=20, min_periods=1).mean().round(4))
# Contamos la cantidad de partidas previas del campeón
df_jugadores['champ_games'] = df_jugadores.groupby('champion')['golddiffat15'].transform(lambda x: x.shift().expanding().count())

# Si el campeón no tiene al menos 20 partidas previas, asignamos 0, si no, su media
df_jugadores['champ_early_power'] = df_jugadores['champ_gold_mean'].where(df_jugadores['champ_games'] >= 20, 0).fillna(0)

# Sumamos el valor de los 5 campeones por equipo
df_comp_early = df_jugadores.groupby(['gameid', 'teamname'])['champ_early_power'].sum().round(4).reset_index()
df_comp_early = df_comp_early.rename(columns={
    'champ_early_power': 'comp_early_power'
})
# ---------------------------------------------------------------------------


# Seleccionamos unicamente las filas resumen del equipo (team)
df_equipos = df_historico[df_historico['position'] == 'team'].copy()


# ----------------- CÁLCULO DE WIN RATIO POR EQUIPO (TEAM_WR) -----------------
# Unificamos MAD Lions KOI y Movistar KOI a KOI para calcular su winratio conjunto
teamnames_unificados = df_equipos['teamname'].replace({'MAD Lions KOI': 'KOI', 'Movistar KOI': 'KOI'})
df_equipos['team_wr'] = df_equipos.groupby(teamnames_unificados)['result'].transform(lambda x: x.shift().ewm(span=20, min_periods=1).mean().round(4))
df_equipos['team_games'] = df_equipos.groupby(teamnames_unificados)['result'].transform(lambda x: x.shift().expanding().count())

# Aplicamos el filtro de 20 partidas para los equipos también, asumiendo su wr en 0.5 si es más bajo
df_equipos['team_wr'] = df_equipos['team_wr'].where(df_equipos['team_games'] >= 20, 0.5).fillna(0.5)
# ---------------------------------------------------------------------------


# Unimos las columnas usando con el win ratio del jugador calculado
df_equipos = pd.merge(df_equipos, df_wr, on=['gameid', 'teamname'], how='left')

# Unimos con los campeones por posición
df_equipos = pd.merge(df_equipos, df_champs, on=['gameid', 'teamname'], how='left')

# Unimos con el win ratio del campeón calculado
df_equipos = pd.merge(df_equipos, df_champ_wr, on=['gameid', 'teamname'], how='left')

# Unimos también el comp_early_power calculado
df_equipos = pd.merge(df_equipos, df_comp_early, on=['gameid', 'teamname'], how='left')

#Seleccionamos las variables útiles para la predicción, sustituyendo los picks por posiciones
variables = ['playoffs', 'side', 'teamname', 'team_wr', 'champ_top', 'champ_jng', 'champ_mid', 'champ_bot', 'champ_sup', 
             'wr_champ_top', 'wr_champ_jng', 'wr_champ_mid', 'wr_champ_bot', 'wr_champ_sup',
             'firstdragon', 'golddiffat15', 'xpdiffat15', 'csdiffat15', 'killsat15', 'assistsat15', 'deathsat15',
             'wr_top', 'wr_jng', 'wr_mid', 'wr_bot', 'wr_sup', 'comp_early_power', 'result'
            ]

# Filtramos el DataFrame para quedarnos solo con las variables seleccionadas
df_equipos = df_equipos[variables]

# Renombramos MAD Lions KOI y Movistar KOI a KOI para unificar su nombre en el DataFrame final
df_equipos['teamname'] = df_equipos['teamname'].replace({'MAD Lions KOI': 'KOI', 'Movistar KOI': 'KOI'})


# Comprobamos que no haya valores nulos en el DataFrame y, si los hay, los eliminamos
if df_equipos.isnull().sum().sum() == 0:
    print("No hay valores nulos en el DataFrame.")
else:
    print("Hay valores nulos en el DataFrame. Se eliminarán las filas con valores nulos.")
    df_equipos = df_equipos.dropna()

# Guardamos el DataFrame filtrado en un nuevo CSV
df_equipos.to_csv('../data/teams/LEC-equipos.csv', index=False)

No hay valores nulos en el DataFrame.
